# Notebook 1 — Data Ingestion & Pre-processing
**Project:** Predicting Oil Well Output Using Hybrid DCA-LSTM  
**Well:** NO 15/9-F-14 H (Volve Oilfield, Norway)  
**Methodology ref:** §3.2.2 (Adefisan Peace Folashade, U22CS1009)

### Steps in this notebook
1. Load Monthly Production Data from the Volve Excel file
2. Filter to F-14H and build a clean date-indexed time series
3. Outlier detection & removal — IQR method (§3.2.2.1)
4. Missing value / zero imputation — linear interpolation (§3.2.2.2)
5. Min-Max normalisation — fit on training set only (§3.2.2.3)
6. Train / Validation / Test split (70 / 15 / 15)
7. Save cleaned artefacts to disk for use in subsequent notebooks

---

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'openpyxl'], check=True)

In [ ]:
import os

# Check where Jupyter thinks you currently are
print("Current working directory:")
print(os.getcwd())

# List all files in that directory so you can see what's there
print("\nFiles in current directory:")
for f in os.listdir('.'):
    print(' ', f)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH   = 'Volve production data.xlsx'   # update path if needed
WELL_NAME   = '15/9-F-14'                   # name as it appears in Monthly sheet
OUTPUT_DIR  = 'pipeline_outputs'            # shared folder for inter-notebook artefacts
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15
# TEST_FRAC is inferred as the remainder (0.15)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration loaded ✓')
print(f'Output directory: {os.path.abspath(OUTPUT_DIR)}')

In [ ]:
pip install --upgrade numexpr

## Step 1 — Load Monthly Production Data

In [ ]:
# Load the Monthly Production Data sheet
# Row 0 of the sheet is a units header row — we skip it after loading
raw = pd.read_excel(DATA_PATH, sheet_name='Monthly Production Data')
raw = raw.iloc[1:].reset_index(drop=True)   # drop units row

# Cast numeric columns
for col in ['Year', 'Month', 'Oil', 'Gas', 'Water', 'On Stream']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

print(f'Total rows in Monthly sheet : {len(raw)}')
print(f'Wells present               : {raw["Wellbore name"].dropna().unique()}')

In [ ]:
# Filter to F-14H and build Date column
df = raw[raw['Wellbore name'] == WELL_NAME].copy()
df = df.dropna(subset=['Year', 'Month'])
df['Date'] = pd.to_datetime(
    df['Year'].astype(int).astype(str) + '-' +
    df['Month'].astype(int).astype(str).str.zfill(2) + '-01'
)
df = df.sort_values('Date').reset_index(drop=True)
df = df[['Date', 'Oil', 'Gas', 'Water', 'On Stream']].rename(columns={
    'Oil'      : 'OIL_PROD',
    'Gas'      : 'GAS_PROD',
    'Water'    : 'WAT_PROD',
    'On Stream': 'ON_STREAM_HRS'
})

print(f'F-14H records   : {len(df)}')
print(f'Date range      : {df["Date"].iloc[0].strftime("%Y-%m")} → {df["Date"].iloc[-1].strftime("%Y-%m")}')
print(f'Zero oil months : {(df["OIL_PROD"] == 0).sum()}')
print(f'Null oil months : {df["OIL_PROD"].isna().sum()}')
df.head(10)

## Step 2 — Remove Leading / Trailing Zero Months

In [ ]:
# Drop leading zero months (before the well came on stream)
first_prod_idx = df[df['OIL_PROD'] > 0].index[0]
df = df.loc[first_prod_idx:].reset_index(drop=True)

# Drop trailing zero months (well shut-in at end of field life)
last_prod_idx = df[df['OIL_PROD'] > 0].index[-1]
df = df.loc[:last_prod_idx].reset_index(drop=True)

print(f'Records after trimming zero tails : {len(df)}')
print(f'Effective production period       : '
      f'{df["Date"].iloc[0].strftime("%Y-%m")} → {df["Date"].iloc[-1].strftime("%Y-%m")}')
print(f'Remaining zero oil months         : {(df["OIL_PROD"] == 0).sum()}')

## Step 3 — Exploratory Visualisation (Raw Data)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(df['Date'], df['OIL_PROD'], color='steelblue', linewidth=1.5)
axes[0].set_ylabel('Oil (Sm³/month)')
axes[0].set_title('NO 15/9-F-14 H — Raw Monthly Production', fontsize=12)
axes[0].grid(alpha=0.3)

axes[1].plot(df['Date'], df['GAS_PROD'], color='darkorange', linewidth=1.5)
axes[1].set_ylabel('Gas (Sm³/month)')
axes[1].grid(alpha=0.3)

axes[2].plot(df['Date'], df['WAT_PROD'], color='teal', linewidth=1.5)
axes[2].set_ylabel('Water (Sm³/month)')
axes[2].set_xlabel('Date')
axes[2].grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01_raw_production.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_01_raw_production.png')

## Step 4 — Outlier Detection & Removal (§3.2.2.1 — IQR Method)

In [ ]:
def detect_outliers_iqr(series: pd.Series,
                         window: int = 12,
                         k: float = 1.5) -> pd.Series:
    """
    Rolling IQR outlier flag.
    Returns a boolean Series: True = outlier.
    Uses a centre-aligned rolling window of `window` months.
    `k` controls the fence width (standard IQR multiplier = 1.5).
    """
    q1 = series.rolling(window, center=True, min_periods=6).quantile(0.25)
    q3 = series.rolling(window, center=True, min_periods=6).quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (series < lower) | (series > upper)


df_clean = df.copy()
outlier_mask = detect_outliers_iqr(df_clean['OIL_PROD'])
n_outliers = outlier_mask.sum()

print(f'Outliers detected (IQR method) : {n_outliers}')
if n_outliers > 0:
    print('Outlier dates:')
    print(df_clean.loc[outlier_mask, ['Date', 'OIL_PROD']].to_string(index=False))

# Flag outliers as NaN for interpolation
df_clean.loc[outlier_mask, 'OIL_PROD'] = np.nan

## Step 5 — Missing Value & Zero Imputation (§3.2.2.2)

In [ ]:
# Also treat internal zero-production months (mid-history shut-ins) as gaps
zero_mask = df_clean['OIL_PROD'] == 0
n_zeros   = zero_mask.sum()
df_clean.loc[zero_mask, 'OIL_PROD'] = np.nan
print(f'Internal zero months flagged for imputation : {n_zeros}')

# Linear interpolation for gaps ≤ 7 months (§3.2.2.2)
df_clean['OIL_PROD'] = df_clean['OIL_PROD'].interpolate(method='linear', limit=7)

# Forward/back fill for any remaining NaN (boundary months)
df_clean['OIL_PROD'] = df_clean['OIL_PROD'].ffill().bfill()

# Ensure no negatives
df_clean['OIL_PROD'] = df_clean['OIL_PROD'].clip(lower=0)

print(f'Null values remaining after imputation : {df_clean["OIL_PROD"].isna().sum()}')
print(f'Final clean records                    : {len(df_clean)}')

In [ ]:
# Visualise: raw vs cleaned
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df['Date'],       df['OIL_PROD'],       color='lightcoral', linewidth=1.2,
        alpha=0.7, label='Raw')
ax.plot(df_clean['Date'], df_clean['OIL_PROD'], color='steelblue',  linewidth=1.8,
        label='Cleaned (outliers + zeros imputed)')

# Mark the outlier / zero positions on the raw series
flagged = outlier_mask | zero_mask
ax.scatter(df.loc[flagged, 'Date'], df.loc[flagged, 'OIL_PROD'],
           color='red', zorder=5, s=40, label='Flagged points')

ax.set_title('Pre-processing: Raw vs Cleaned — NO 15/9-F-14 H', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Oil Production (Sm³/month)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_02_raw_vs_clean.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_02_raw_vs_clean.png')

## Step 6 — Descriptive Statistics of Clean Data

In [ ]:
stats = df_clean['OIL_PROD'].describe()
print('Cleaned oil production descriptive statistics:')
print(stats.to_string())
print(f'\nSkewness  : {df_clean["OIL_PROD"].skew():.4f}')
print(f'Kurtosis  : {df_clean["OIL_PROD"].kurtosis():.4f}')

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_clean['OIL_PROD'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Oil Production Distribution', fontsize=11)
axes[0].set_xlabel('Sm³/month')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

axes[1].boxplot(df_clean['OIL_PROD'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Box Plot — Oil Production', fontsize=11)
axes[1].set_ylabel('Sm³/month')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_03_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_03_distribution.png')

## Step 7 — Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
n       = len(df_clean)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)
n_test  = n - n_train - n_val

df_train = df_clean.iloc[:n_train].reset_index(drop=True)
df_val   = df_clean.iloc[n_train : n_train + n_val].reset_index(drop=True)
df_test  = df_clean.iloc[n_train + n_val :].reset_index(drop=True)

print(f'Total records  : {n}')
print(f'Train ({TRAIN_FRAC:.0%}) : {len(df_train)} months  '
      f'({df_train["Date"].iloc[0].strftime("%Y-%m")} → {df_train["Date"].iloc[-1].strftime("%Y-%m")})')
print(f'Val   ({VAL_FRAC:.0%}) : {len(df_val)} months  '
      f'({df_val["Date"].iloc[0].strftime("%Y-%m")} → {df_val["Date"].iloc[-1].strftime("%Y-%m")})')
print(f'Test  ({1-TRAIN_FRAC-VAL_FRAC:.0%}) : {len(df_test)} months  '
      f'({df_test["Date"].iloc[0].strftime("%Y-%m")} → {df_test["Date"].iloc[-1].strftime("%Y-%m")})')

In [ ]:
# Visualise split
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df_train['Date'], df_train['OIL_PROD'], color='steelblue',  lw=1.8, label='Train')
ax.plot(df_val['Date'],   df_val['OIL_PROD'],   color='darkorange', lw=1.8, label='Validation')
ax.plot(df_test['Date'],  df_test['OIL_PROD'],  color='red',        lw=1.8, label='Test')
ax.axvline(df_val['Date'].iloc[0],  color='gray',  linestyle='--', lw=1)
ax.axvline(df_test['Date'].iloc[0], color='black', linestyle='--', lw=1)
ax.set_title('Train / Validation / Test Split — NO 15/9-F-14 H', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Oil Production (Sm³/month)')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_04_train_val_test_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_04_train_val_test_split.png')

## Step 8 — Min-Max Normalisation (§3.2.2.3)
> **Important:** The scaler is fitted **only on the training set** and then applied to val and test — preventing data leakage.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df_train[['OIL_PROD']])

print(f'Scaler fitted on training set only')
print(f'  Training min : {scaler.data_min_[0]:,.2f} Sm³/month')
print(f'  Training max : {scaler.data_max_[0]:,.2f} Sm³/month')

## Step 9 — Save All Artefacts for Subsequent Notebooks

In [ ]:
# Save split DataFrames as CSV
df_train.to_csv(f'{OUTPUT_DIR}/f14h_train.csv', index=False)
df_val.to_csv(  f'{OUTPUT_DIR}/f14h_val.csv',   index=False)
df_test.to_csv( f'{OUTPUT_DIR}/f14h_test.csv',  index=False)
df_clean.to_csv(f'{OUTPUT_DIR}/f14h_clean.csv', index=False)

# Save scaler
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save split sizes for downstream use
split_info = {'n_train': n_train, 'n_val': n_val, 'n_test': n_test, 'n_total': n}
with open(f'{OUTPUT_DIR}/split_info.pkl', 'wb') as f:
    pickle.dump(split_info, f)

print('Artefacts saved to pipeline_outputs/:')
print('  f14h_clean.csv   — full clean time series')
print('  f14h_train.csv   — training partition')
print('  f14h_val.csv     — validation partition')
print('  f14h_test.csv    — test partition')
print('  scaler.pkl       — fitted MinMaxScaler')
print('  split_info.pkl   — split index sizes')
print()
print('Notebook 1 complete ✓  →  proceed to 02_DCA_Fitting.ipynb')